In [35]:
# Required imports
from sklearn.model_selection import GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.neural_network import MLPClassifier

from sklearn.pipeline import Pipeline

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.svm import SVR
from sklearn.metrics import mean_squared_error,mean_absolute_error, r2_score


In [28]:

# 1. Logistic Regression
logreg = LogisticRegression(max_iter=5000)
logreg_params = {
    "C": [0.01, 0.1, 1, 10, 100],
    "penalty": ["l2"],          # 'l1' requires solver='liblinear' or 'saga'
    "solver": ["lbfgs", "saga"]
}

# 2. Support Vector Classifier
svc = SVC()
svc_params = {
    "kernel": ["rbf", "poly", "sigmoid"],
    "C": [0.1, 1, 10, 100],
    "gamma": ["scale", "auto"]
}

# 3. K-Nearest Neighbors
knn = KNeighborsClassifier()
knn_params = {
    "n_neighbors": [3, 5, 7, 9],
    "weights": ["uniform", "distance"],
    "metric": ["euclidean", "manhattan"]
}

# 4. Decision Tree Classifier
dt = DecisionTreeClassifier()
dt_params = {
    "max_depth": [None, 5, 10, 20],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 4]
}

# 5. Random Forest Classifier
rf = RandomForestClassifier(random_state=42)
rf_params = {
    "n_estimators": [50, 100, 200],
    "max_depth": [None, 5, 10, 20],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 4]
}

# 6. Gradient Boosting Classifier
gbc = GradientBoostingClassifier()
gbc_params = {
    "n_estimators": [50, 100, 200],
    "learning_rate": [0.01, 0.05, 0.1, 0.2],
    "max_depth": [3, 5],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 4]
}

# 7. Neural Network (MLPClassifier)
mlp = MLPClassifier(max_iter=3000)
mlp_params = {
    "hidden_layer_sizes": [(50,), (100,), (100, 50)],
    "activation": ["relu", "tanh"],
    "alpha": [0.0001, 0.001, 0.01],
    "learning_rate_init": [0.001, 0.01]
}


# Combine them into your expected dictionary:
models = {
    "LogisticRegression": (logreg, logreg_params),
    "SVC": (svc, svc_params),
    "KNN": (knn, knn_params),
    "DecisionTree": (dt, dt_params),
    "RandomForest": (rf, rf_params),
 #   "GradientBoosting": (gbc, gbc_params),
    "MLPClassifier": (mlp, mlp_params),
}



In [93]:
def build_X_keys_And_Y_target_Arrays(userInputData,closeDoorTraining_openDoorTest = False)->(np.array,np.array):
  
    if closeDoorTraining_openDoorTest :
         pass

    else:
        keys = np.array(userInputData.index)
        keys_numpy = keys.reshape(-1,1)
   

        # columns_to_pass = ["Euclidian distance to Id="+str(id)+":BME680:breathVocEquivalent"
        #                            for id in range(3) ]
        # column_size = len(columns_to_pass)
        # target_variables = userInputData.loc[keys,columns_to_pass].to_numpy()
        # target_variables_euclidian_distance_numpy = target_variables.reshape(-1,column_size)
     
    
    
        columns_to_pass = ["x-y axis"]
        flattened = userInputData.loc[keys,columns_to_pass].to_numpy().ravel()
        Label_Encoder = LabelEncoder()
                # Step 1: flatten the array from shape (98, 1) of tuples to (98,) of tuples
        
        # Step 2: convert each tuple to a string "x_y"
        string_labels = np.array([f"{x}_{y}" for (x, y) in flattened])
        
        # Step 3: reshape to (98, 1)
        string_labels = string_labels.reshape(-1, 1)
        target_variables_positions_numpy =  Label_Encoder.fit_transform(string_labels) 
        
        
        X_train_keys,X_test_keys,Y_train,Y_test = train_test_split(
            keys_numpy,target_variables_positions_numpy,
            test_size=0.2,
            stratify=target_variables_positions_numpy,
            random_state=42
        )
    
    return X_train_keys,X_test_keys,Y_train,Y_test,Label_Encoder

#### PLOT CONFUSION MATRIX

In [44]:
def plotConfusionMatrix(y_test, y_pred,Label_Encoder):
    # encoder used earlier
    labels = le.classes_                # encoded labels in correct order
    decoded_labels = [str(x) for x in labels]  # tuples → strings for display
    
    cm = confusion_matrix(y_test, y_pred, labels=range(len(labels)))
    
    disp = ConfusionMatrixDisplay(
        confusion_matrix=cm,
        display_labels=decoded_labels
    )
    
    disp.plot(cmap='Blues', values_format='d')
    plt.xticks(rotation=45)
    plt.yticks(rotation=0)
    plt.tight_layout()
    plt.show()

#### GET SCORES

In [45]:
def getScores(y_test, y_pred):
    accuracy = accuracy_score(y_test, y_pred)
    
    precision = precision_score(
        y_test, y_pred,
        average='weighted'   # or 'macro', 'micro'
    )
    
    recall = recall_score(
        y_test, y_pred,
        average='weighted'
    )
    
    print(f"Accuracy : {accuracy:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall   : {recall:.4f}")
    return accuracy,precision,recall